# Clinical-Only Data Preparation

This notebook prepares model-ready data using **only clinical features** (no machine pressure data):
1. **Lab features** (longitudinal daily lab values) from `lab_transformed_longitudinal.csv`
2. **PTI features** (patient-level demographics/comorbidities) from `pti_transformed_cleaned.csv`

### Key decisions:
- **No date columns** used as features
- **No machine features** — clinical data only
- **CrrtInitiationWeight_kg** used as the weight feature (HospitalWeight_kg and IcuWeight_kg dropped)
- **Lab missing values**: forward-fill then backward-fill per patient (no imputation)
- **Features with >40% missing after fill** are automatically dropped
- **Existing PID-based train/val/test splits** are preserved
- Output: model-ready CSVs for 1day and 2day prediction horizons

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---- Paths ----
PROJ_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR  = os.path.join(PROJ_DIR, 'Data')

LAB_PATH  = os.path.join(DATA_DIR, 'EHR', 'Lab', 'lab_transformed_longitudinal.csv')
PTI_PATH  = os.path.join(DATA_DIR, 'EHR', 'Demographics', 'pti_transformed_cleaned.csv')
MODEL_READY_DIR = os.path.join(DATA_DIR, 'Processed', 'Model_Ready')
LABELS_DIR = os.path.join(DATA_DIR, 'Processed', 'Labels')
SPLITS_PATH = os.path.join(MODEL_READY_DIR, 'splits', 'pid_splits.json')

OUTPUT_DIR = os.path.join(MODEL_READY_DIR, 'clinical_only')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Project dir: {PROJ_DIR}')
print(f'Output dir:  {OUTPUT_DIR}')

## 1. Load Raw Data

In [ ]:
# Load clinical data
lab_raw = pd.read_csv(LAB_PATH)
pti_raw = pd.read_csv(PTI_PATH)

# Load PID splits
with open(SPLITS_PATH) as f:
    pid_splits = json.load(f)

print(f'Lab data:  {lab_raw.shape}')
print(f'PTI data:  {pti_raw.shape}')
print(f'Train PIDs: {len(pid_splits["train"])}, Val PIDs: {len(pid_splits["val"])}, Test PIDs: {len(pid_splits["test"])}')

## 2. Process PTI Data (Patient-Level Features)

- Drop date/time columns (not features)
- Drop FormDataId (identifier, not feature)
- Drop HospitalWeight_kg and IcuWeight_kg (use CrrtInitiationWeight_kg)
- Keep PID as merge key

In [ ]:
# Columns to drop from PTI
PTI_DROP_COLS = [
    'FormDataId',
    # Date/time columns
    'HospitalAdmitDate', 'IcuAdmitDate', 'CrrtStartDate',
    'CrrtStartTime_24HourFormat', 'CrrtStartTime_FractionOfDay',
    # Redundant weight columns
    'HospitalWeight_kg', 'IcuWeight_kg',
]

pti = pti_raw.drop(columns=PTI_DROP_COLS, errors='ignore')

pti_feature_cols = [c for c in pti.columns if c != 'PID']
print(f'PTI features: {len(pti_feature_cols)} columns')
print(f'PTI shape: {pti.shape}')
print(f'\nPTI missing values (non-zero):')
missing = pti[pti_feature_cols].isnull().sum()
print(missing[missing > 0])

## 3. Process Lab Data (Day-Level Features)

Lab data is longitudinal (multiple rows per PID per date). Steps:
1. Drop non-feature columns (FormDataId, TimeOfDay24Hour)
2. Aggregate multiple measurements per PID+Date by taking the **mean**
3. Sort by PID and date, then **forward-fill** + **backward-fill** missing values per PID

In [ ]:
# Lab value columns (exclude PID, FormDataId, date, time)
LAB_DROP_COLS = ['FormDataId', 'TimeOfDay24Hour']
lab = lab_raw.drop(columns=LAB_DROP_COLS)

lab_value_cols = [c for c in lab.columns if c not in ['PID', 'DailyLaboratoryDataDate']]
print(f'Lab value columns ({len(lab_value_cols)}):')
print(lab_value_cols)

In [ ]:
# Step 1: Aggregate multiple measurements per PID+Date (mean)
lab_daily = lab.groupby(['PID', 'DailyLaboratoryDataDate'])[lab_value_cols].mean().reset_index()
lab_daily = lab_daily.sort_values(['PID', 'DailyLaboratoryDataDate']).reset_index(drop=True)

print(f'Lab daily (after aggregation): {lab_daily.shape}')
print(f'Missing BEFORE fill:')
print(lab_daily[lab_value_cols].isnull().sum())

In [ ]:
# Step 2: Forward-fill then backward-fill per PID
lab_daily[lab_value_cols] = lab_daily.groupby('PID')[lab_value_cols].transform(
    lambda grp: grp.ffill().bfill()
)

print(f'Missing AFTER fill:')
print(lab_daily[lab_value_cols].isnull().sum())
print(f'\nNote: Remaining NaNs are PIDs with ALL missing values for that lab.')

## 4. Build Clinical-Only Dataset

Use the labels file to create the day-level structure (PID, Visit_no, Day, label),
then merge lab (by PID+Date) and PTI (by PID) features.

In [ ]:
MISS_THRESHOLD = 0.40  # drop features with >40% missing after fill

def process_case(case_name):
    """
    For a given prediction case (1day, 2day), build clinical-only dataset
    by merging lab + PTI features with labels, using existing PID splits.
    
    Returns dict with train_df, val_df, test_df.
    """
    print(f'\n{"="*60}')
    print(f'Processing case: {case_name} (clinical only)')
    print(f'{"="*60}')
    
    # --- Load labels (has PID, Visit_no, Day, Date, label) ---
    labels = pd.read_csv(os.path.join(LABELS_DIR, f'labels_{case_name}.csv'))
    labels['Date'] = pd.to_datetime(labels['Date'])
    # Rename label column to generic 'label'
    label_col = f'label_{case_name}'
    labels = labels.rename(columns={label_col: 'label'})
    print(f'Labels: {labels.shape}')
    print(f'  Positive labels: {labels["label"].sum()} / {len(labels)} ({100*labels["label"].mean():.1f}%)')
    
    # --- Merge lab data by PID + Date ---
    lab_merge = lab_daily.copy()
    lab_merge['DailyLaboratoryDataDate'] = pd.to_datetime(lab_merge['DailyLaboratoryDataDate'])
    lab_merge = lab_merge.rename(columns={'DailyLaboratoryDataDate': 'Date'})
    
    # Prefix lab columns
    lab_rename = {c: f'LAB_{c}' for c in lab_value_cols}
    lab_merge = lab_merge.rename(columns=lab_rename)
    lab_feat_cols = [f'LAB_{c}' for c in lab_value_cols]
    
    merged = labels.merge(
        lab_merge[['PID', 'Date'] + lab_feat_cols],
        on=['PID', 'Date'], how='left'
    )
    
    n_with_lab = merged[lab_feat_cols[0]].notna().sum()
    print(f'Rows with lab data: {n_with_lab}/{len(merged)} ({100*n_with_lab/len(merged):.1f}%)')
    
    # --- Forward-fill + backward-fill lab features per PID within merged data ---
    merged = merged.sort_values(['PID', 'Visit_no', 'Day']).reset_index(drop=True)
    merged[lab_feat_cols] = merged.groupby('PID')[lab_feat_cols].transform(
        lambda grp: grp.ffill().bfill()
    )
    
    n_with_lab_after = merged[lab_feat_cols[0]].notna().sum()
    print(f'Rows with lab data after fill: {n_with_lab_after}/{len(merged)} ({100*n_with_lab_after/len(merged):.1f}%)')
    
    # --- Merge PTI data by PID ---
    pti_rename = {c: f'PTI_{c}' for c in pti_feature_cols}
    pti_merge = pti.rename(columns=pti_rename)
    pti_feat_cols = [f'PTI_{c}' for c in pti_feature_cols]
    
    merged = merged.merge(
        pti_merge[['PID'] + pti_feat_cols],
        on='PID', how='left'
    )
    
    n_with_pti = merged[pti_feat_cols[0]].notna().sum()
    print(f'Rows with PTI data: {n_with_pti}/{len(merged)} ({100*n_with_pti/len(merged):.1f}%)')
    
    # --- Drop features with >40% missing after all filling ---
    all_clinical_cols = lab_feat_cols + pti_feat_cols
    miss_pct = merged[all_clinical_cols].isnull().mean()
    high_miss = miss_pct[miss_pct > MISS_THRESHOLD]
    
    if len(high_miss) > 0:
        print(f'\nDropping {len(high_miss)} features with >{MISS_THRESHOLD*100:.0f}% missing after fill:')
        for col, pct in high_miss.sort_values(ascending=False).items():
            print(f'  {col}: {pct*100:.1f}%')
        
        dropped_lab = [c for c in high_miss.index if c in lab_feat_cols]
        dropped_pti = [c for c in high_miss.index if c in pti_feat_cols]
        lab_feat_cols = [c for c in lab_feat_cols if c not in high_miss.index]
        pti_feat_cols = [c for c in pti_feat_cols if c not in high_miss.index]
        merged = merged.drop(columns=high_miss.index.tolist())
        print(f'  Removed: {len(dropped_lab)} lab, {len(dropped_pti)} PTI')
    else:
        print(f'\nNo features exceed {MISS_THRESHOLD*100:.0f}% missing threshold.')
    
    # --- Define final feature columns ---
    all_feature_cols = lab_feat_cols + pti_feat_cols
    print(f'\nTotal features: {len(all_feature_cols)}')
    print(f'  Lab: {len(lab_feat_cols)}')
    print(f'  PTI: {len(pti_feat_cols)}')
    
    # --- Split into train/val/test using existing PID splits ---
    train_pids = set(pid_splits['train'])
    val_pids   = set(pid_splits['val'])
    test_pids  = set(pid_splits['test'])
    
    meta_cols = ['PID', 'Visit_no', 'Day', 'Mortality', 'days_until_death', 'label']
    out_cols = meta_cols + all_feature_cols
    
    train_df = merged[merged['PID'].isin(train_pids)][out_cols].reset_index(drop=True)
    val_df   = merged[merged['PID'].isin(val_pids)][out_cols].reset_index(drop=True)
    test_df  = merged[merged['PID'].isin(test_pids)][out_cols].reset_index(drop=True)
    
    print(f'\nSplit sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    print(f'Label distribution:')
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        n_pos = df['label'].sum()
        print(f'  {name}: {n_pos} pos / {len(df)} total ({100*n_pos/len(df):.1f}%)')
    
    # --- Check remaining missing values ---
    print(f'\nRemaining missing values in features (non-zero):')
    miss = train_df[all_feature_cols].isnull().sum()
    miss_nz = miss[miss > 0]
    if len(miss_nz) > 0:
        print(miss_nz)
    else:
        print('  None!')
    
    return {
        'train': train_df,
        'val': val_df,
        'test': test_df,
        'feature_cols': all_feature_cols,
        'lab_feat_cols': lab_feat_cols,
        'pti_feat_cols': pti_feat_cols,
    }

In [ ]:
# Process both cases
results = {}
for case_name in ['1day', '2day']:
    results[case_name] = process_case(case_name)

## 5. Save Model-Ready Data

In [ ]:
for case_name, data in results.items():
    case_dir = os.path.join(OUTPUT_DIR, case_name)
    os.makedirs(case_dir, exist_ok=True)
    
    for split_name in ['train', 'val', 'test']:
        out_path = os.path.join(case_dir, f'{split_name}.csv')
        data[split_name].to_csv(out_path, index=False)
        print(f'Saved: {out_path}  ({data[split_name].shape})')
    
    # Save feature column names for reference
    feat_info = {
        'all_features': data['feature_cols'],
        'lab_features': data['lab_feat_cols'],
        'pti_features': data['pti_feat_cols'],
    }
    feat_path = os.path.join(case_dir, 'feature_info.json')
    with open(feat_path, 'w') as f:
        json.dump(feat_info, f, indent=2)
    print(f'Saved: {feat_path}')
    print()

## 6. Feature Summary

In [ ]:
# Summary statistics for the clinical features
for case_name, data in results.items():
    print(f'\n{"="*60}')
    print(f'Case: {case_name}')
    print(f'{"="*60}')
    
    train = data['train']
    feat_cols = data['feature_cols']
    
    print(f'Total features: {len(feat_cols)}')
    print(f'  Lab: {len(data["lab_feat_cols"])}')
    print(f'  PTI: {len(data["pti_feat_cols"])}')
    print(f'\nTrain shape: {train.shape}')
    print(f'\nFeature stats (train):')
    display(train[feat_cols].describe().T)

In [ ]:
# Missing value plot
import matplotlib.pyplot as plt
%matplotlib inline

for case_name, data in results.items():
    train = data['train']
    feat_cols = data['feature_cols']
    
    miss_pct = train[feat_cols].isnull().mean() * 100
    miss_nz = miss_pct[miss_pct > 0].sort_values(ascending=False)
    
    if len(miss_nz) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(miss_nz)*0.3)))
        miss_nz.plot.barh(ax=ax, color='steelblue')
        ax.set_xlabel('Missing %')
        ax.set_title(f'{case_name} - Remaining Missing Values After Fill (Clinical Only)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, case_name, 'missing_values.png'), dpi=100)
        plt.show()
    else:
        print(f'{case_name}: No missing values in training features!')

In [ ]:
print('\nData preparation complete!')
print(f'Output directory: {OUTPUT_DIR}')
print(f'\nTo train models, run:')
print(f'  cd Baseline_ML_Models/Clinical_Only_Models')
print(f'  python run_experiments.py')